In [1]:
import sys
import os
import torch
from torch import nn
from vggt.models.vggt import VGGT
from transformers import AutoModelForImageTextToText, AutoProcessor
from PIL import Image

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
device = "cuda"
dtype = torch.bfloat16

# Import Tokenizer & Model
---

In [3]:
processor = AutoProcessor.from_pretrained("/home/ubuntu/Shree_FYP/data/stage1_unsloth")

[transformers] The tokenizer you are loading from '/home/ubuntu/Shree_FYP/data/stage1_unsloth' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [4]:
vlm = AutoModelForImageTextToText.from_pretrained(
    "unsloth/Qwen3.5-4B",
    dtype = torch.bfloat16,
    device_map = "cuda"
)

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [5]:
def extract_vision_hidden(
    model,
    processor,
    prompt,
    layer_idx=24,
    device=None,
):
    if device is None:
        device = next(model.parameters()).device

    prompt = {
        k: v.to(device) if torch.is_tensor(v) else v
        for k, v in prompt.items()
    }

    with torch.no_grad():
        output = model(
            **prompt,
            output_hidden_states=True,
            use_cache=False,
        )

    hidden = output.hidden_states[layer_idx]
    input_ids = prompt["input_ids"]

    image_token_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
    video_token_id = processor.tokenizer.convert_tokens_to_ids("<|video_pad|>")

    image_mask = input_ids == image_token_id
    video_mask = input_ids == video_token_id

    num_image_tokens = image_mask.sum().item()
    num_video_tokens = video_mask.sum().item()

    if num_image_tokens > 0 and num_video_tokens > 0:
        raise ValueError(
            "Both <|image_pad|> and <|video_pad|> were found in input_ids. "
            "This function expects one media type per sample."
        )

    if num_image_tokens > 0:
        media_type = "image"
        media_mask = image_mask
    elif num_video_tokens > 0:
        media_type = "video"
        media_mask = video_mask
    else:
        raise ValueError(
            "No media tokens found. Expected either <|image_pad|> or "
            "<|video_pad|> inside input_ids."
        )

    if hidden.shape[0] != 1:
        raise ValueError(
            f"Expected batch size 1 since you're sending sample by sample, "
            f"but got batch size {hidden.shape[0]}."
        )

    media_hidden = hidden[0, media_mask[0], :]

    return media_hidden

In [6]:
ckpt = torch.load(
    "/home/ubuntu/Shree_FYP/model.pt",
    map_location="cpu",
)

vggt_model = VGGT(
    enable_camera=False,
    enable_point=False,
    enable_depth=False,
    enable_track=False,
    feature_only=True
).eval()    

vggt_model = vggt_model.to(dtype=dtype)
vggt_model.load_state_dict(ckpt, strict=False)
vggt_model = vggt_model.to(device)

In [7]:
class AlignProjector(nn.Module):
    """
    Projects Qwen3.5 4B vision embeddings (2560) to VGGT embeddings (2048) 
    and computes the Cosine Alignment Loss.
    """
    def __init__(
            self,
            llm_dim: int = 2560,     # Qwen3.5 4B hidden size
            vggt_dim: int = 2048,    # Your specific VGGT embed_dim
            align_loss_type: str = "cosine",
            use_vlm_norm: bool = False,
        ) -> None:
        super().__init__()
        self.llm_dim = llm_dim
        self.vggt_dim = vggt_dim
        self.align_loss_type = align_loss_type

        # Bottleneck layer to prevent overfitting and save VRAM
        # Max of (2048, 2560//2) = 2048
        hidden_dim = max(self.vggt_dim, self.llm_dim // 2) 
        
        self.fc1 = nn.Linear(self.llm_dim, hidden_dim, bias=True)
        self.fc2 = nn.Linear(hidden_dim, self.vggt_dim, bias=True) # Maps exactly to 2048
        self.act_fn1 = nn.GELU()
        
        self.vlm_norm = nn.LayerNorm(llm_dim) if use_vlm_norm else None
        self.initialize_weights()
    
    def initialize_weights(self):
        def _basic_init(module):
            if isinstance(module, nn.Linear):
                torch.nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
        self.apply(_basic_init)

    def align_dimension(self, LLM_embedding: torch.Tensor) -> torch.Tensor:
        if self.vlm_norm is not None:
            LLM_embedding = self.vlm_norm(LLM_embedding)
        projected_features = self.fc1(LLM_embedding)
        projected_features = self.act_fn1(projected_features)
        projected_features = self.fc2(projected_features)
        return projected_features
    
    def compute_align_loss_cosine(self, vision_hidden, vggt_hidden):
        align_loss = 0.0
        bsz = len(vision_hidden)
        
        for _vision, _vggt in zip(vision_hidden, vggt_hidden):
            _vision = torch.nn.functional.normalize(_vision, dim=-1)
            _vggt = torch.nn.functional.normalize(_vggt, dim=-1)
            
            # Dot product over the feature dimension, then mean over the sequence length
            cosine_sim = (_vision * _vggt).sum(dim=-1)
            align_loss += 1.0 - cosine_sim.mean() 
            
        align_loss /= bsz
        return align_loss
    
    def forward(self, LLM_emb, target_emb):
        if self.align_loss_type == "cosine":
            # Project in bf16 to save VRAM
            with torch.autocast("cuda", dtype=torch.bfloat16):
                LLM_emb = self.align_dimension(LLM_emb)
                
            # CRITICAL: Cast to fp32 for cosine similarity to avoid underflow/NaNs in bf16
            align_loss = self.compute_align_loss_cosine(LLM_emb.float(), target_emb.float())
            return align_loss
        else:
            raise NotImplementedError(f"Align loss type {self.align_loss_type} is not implemented.")

In [8]:
align_projector = AlignProjector(
    llm_dim=2560,   # Qwen
    vggt_dim=2048,  # Your VGGT
    align_loss_type="cosine",
    use_vlm_norm=False 
).to(device)

---
## Test Batch Padding & Masking
---

In [9]:
frames_path_1 = "/home/ubuntu/Shree_FYP/data/ShareRobot/planning/images/rt_frames_success/rtx_frames_success_0/10_utokyo_pr2_tabletop_manipulation_converted_externally_to_rlds#episode_1"
images_paths_1 = [
    os.path.join(frames_path_1, f) 
    for f in os.listdir(frames_path_1) 
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
]
images_paths_1.sort()
images_1 = [Image.open(img_path) for img_path in images_paths_1]

In [10]:
frames_path_2 = "/home/ubuntu/Shree_FYP/data/ShareRobot/planning/images/rt_frames_success/rtx_frames_success_1/25_nyu_door_opening_surprising_effectiveness#episode_54"
images_paths_2 = [
    os.path.join(frames_path_2, f) 
    for f in os.listdir(frames_path_2) 
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
]
images_paths_2.sort()
images_2 = [Image.open(img_path) for img_path in images_paths_2]

In [11]:
image_3 = Image.open("/home/ubuntu/Shree_FYP/train/stage2/test/images/cat.png")

In [12]:
messages = [
    [
        {
            "role": "user",
            "content": [
                {"type": "video", "video": images_1},
                {"type": "text", "text": "Describe this video?"}
            ]
        }
    ],
    [
        {
            "role": "user",
            "content": [
                {"type": "video", "video": images_2},
                {"type": "text", "text": "What's happening in this?"}
            ]
        }
    ],
    [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_3},
                {"type": "text", "text": "What breed is this cat?"}
            ]
        }
    ]
    
]

In [13]:
messages

[[{'role': 'user',
   'content': [{'type': 'video',
     'video': [<PIL.PngImagePlugin.PngImageFile image mode=RGB size=128x128>,
      <PIL.PngImagePlugin.PngImageFile image mode=RGB size=128x128>,
      <PIL.PngImagePlugin.PngImageFile image mode=RGB size=128x128>,
      <PIL.PngImagePlugin.PngImageFile image mode=RGB size=128x128>,
      <PIL.PngImagePlugin.PngImageFile image mode=RGB size=128x128>,
      <PIL.PngImagePlugin.PngImageFile image mode=RGB size=128x128>,
      <PIL.PngImagePlugin.PngImageFile image mode=RGB size=128x128>,
      <PIL.PngImagePlugin.PngImageFile image mode=RGB size=128x128>,
      <PIL.PngImagePlugin.PngImageFile image mode=RGB size=128x128>,
      <PIL.PngImagePlugin.PngImageFile image mode=RGB size=128x128>,
      <PIL.PngImagePlugin.PngImageFile image mode=RGB size=128x128>,
      <PIL.PngImagePlugin.PngImageFile image mode=RGB size=128x128>,
      <PIL.PngImagePlugin.PngImageFile image mode=RGB size=128x128>,
      <PIL.PngImagePlugin.PngImageFile ima

In [14]:
inputs_text = processor.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True
)

In [15]:
print(inputs_text)

['<|im_start|>user\n<|vision_start|><|video_pad|><|vision_end|>Describe this video?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n', "<|im_start|>user\n<|vision_start|><|video_pad|><|vision_end|>What's happening in this?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n", '<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>What breed is this cat?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n']


In [16]:
inputs = processor.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_dict = True,
    return_tensors = "pt",
    processor_kwargs = {
        "padding": True
    }
).to("cuda")

[transformers] Qwen3VL requires frame timestamps to construct prompts, but the `fps` of the input video could not be inferred. Probably `video_metadata` was missing from inputs and you passed pre-sampled frames. Defaulting to `fps=24`. Please provide `video_metadata` for more accurate results.


In [17]:
processor.decode(inputs.input_ids[0])

'<|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vis

In [18]:
processor.decode(inputs.input_ids[1])

"<|im_start|>user\n<0.0 seconds><|vision_start|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|v

In [19]:
processor.decode(inputs.input_ids[2])

'<|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vis

In [20]:
inputs.attention_mask[0].sum(), inputs.attention_mask[1].sum(), inputs.attention_mask[2].sum()

(tensor(280, device='cuda:0'),
 tensor(5362, device='cuda:0'),
 tensor(1670, device='cuda:0'))

In [21]:
from PIL import Image


def extract_visuals_from_sample(sample_messages):
    """
    Extract all PIL images/frames from one chat sample.

    sample_messages example:
        [
            {
                "role": "user",
                "content": [
                    {"type": "video", "video": [PIL, PIL, ...]},
                    {"type": "text", "text": "..."}
                ]
            }
        ]

    Returns:
        List[PIL.Image]
    """

    visuals = []

    for message in sample_messages:
        for item in message.get("content", []):
            item_type = item.get("type")

            if item_type == "video":
                frames = item["video"]

                if isinstance(frames, Image.Image):
                    frames = [frames]

                visuals.extend(frames)

            elif item_type == "image":
                image = item["image"]

                # Your image field may be a single PIL image or a list containing one PIL image
                if isinstance(image, Image.Image):
                    visuals.append(image)
                elif isinstance(image, list):
                    visuals.extend(image)
                else:
                    raise TypeError(f"Unexpected image type: {type(image)}")

    if len(visuals) == 0:
        raise ValueError("No image/video frames found in this sample.")

    return visuals

In [22]:
import torch
import torch.nn.functional as F
from torchvision.transforms import functional as TVF


def preprocess_pil_image_for_vggt(
    image,
    mode="crop",
    target_size=518,
    dtype=torch.float32,
    device=None,
):
    """
    Convert one PIL image/frame into VGGT format.

    Input:
        PIL image

    Output:
        Tensor [3, 518, 518], values in [0, 1]
    """

    if mode not in ["crop", "pad"]:
        raise ValueError("mode must be either 'crop' or 'pad'")

    image = image.convert("RGB")

    # PIL [H, W, 3], values 0-255
    # Tensor [3, H, W], values 0.0-1.0
    image = TVF.to_tensor(image).to(dtype=dtype)

    if device is not None:
        image = image.to(device)

    _, H, W = image.shape

    if mode == "crop":
        # Resize so the shorter side reaches target_size,
        # then center-crop to target_size x target_size.
        scale = target_size / min(H, W)

        new_h = round(H * scale / 14) * 14
        new_w = round(W * scale / 14) * 14

        new_h = max(new_h, target_size)
        new_w = max(new_w, target_size)

        image = F.interpolate(
            image.unsqueeze(0),
            size=(new_h, new_w),
            mode="bicubic",
            align_corners=False,
        ).squeeze(0)

        top = (new_h - target_size) // 2
        left = (new_w - target_size) // 2

        image = image[:, top:top + target_size, left:left + target_size]

    else:
        # Resize so the longer side reaches target_size,
        # then pad to target_size x target_size.
        scale = target_size / max(H, W)

        new_h = round(H * scale / 14) * 14
        new_w = round(W * scale / 14) * 14

        new_h = max(new_h, 14)
        new_w = max(new_w, 14)

        image = F.interpolate(
            image.unsqueeze(0),
            size=(new_h, new_w),
            mode="bicubic",
            align_corners=False,
        ).squeeze(0)

        pad_h = target_size - new_h
        pad_w = target_size - new_w

        pad_top = pad_h // 2
        pad_bottom = pad_h - pad_top
        pad_left = pad_w // 2
        pad_right = pad_w - pad_left

        image = F.pad(
            image,
            pad=(pad_left, pad_right, pad_top, pad_bottom),
            mode="constant",
            value=1.0,  # white image padding
        )

    return image

In [23]:
def preprocess_padded_visual_batch_for_vggt(
    batch_messages,
    mode="crop",
    target_size=518,
    max_frames=None,
    pad_frame_value=1.0,
    dtype=torch.float32,
    device=None,
):
    """
    Preprocess a mixed image/video chat batch for VGGT.

    Args:
        batch_messages:
            List of samples.
            Each sample is your chat-format list of messages.

        max_frames:
            If None, use the maximum number of frames/images in this batch.
            If set, truncate each sample to at most max_frames.

    Returns:
        vggt_imgs:
            [B, max_N, 3, 518, 518]

        frame_mask:
            [B, max_N], True for real frame/image, False for padding

        frame_lengths:
            [B], number of real frames/images per sample
    """

    visual_lists = []

    for sample_messages in batch_messages:
        visuals = extract_visuals_from_sample(sample_messages)

        if max_frames is not None:
            visuals = visuals[:max_frames]

        visual_lists.append(visuals)

    frame_lengths = torch.tensor(
        [len(v) for v in visual_lists],
        dtype=torch.long,
        device=device,
    )

    B = len(visual_lists)
    max_N = int(frame_lengths.max().item())

    vggt_imgs = torch.full(
        (B, max_N, 3, target_size, target_size),
        fill_value=pad_frame_value,
        dtype=dtype,
        device=device,
    )

    frame_mask = torch.zeros(
        (B, max_N),
        dtype=torch.bool,
        device=device,
    )

    for b, visuals in enumerate(visual_lists):
        for n, image in enumerate(visuals):
            image_tensor = preprocess_pil_image_for_vggt(
                image,
                mode=mode,
                target_size=target_size,
                dtype=dtype,
                device=device,
            )

            vggt_imgs[b, n] = image_tensor
            frame_mask[b, n] = True

    return vggt_imgs, frame_mask, frame_lengths

In [24]:
def pad_hidden_list(hidden_list, padding_value=0.0):
    """
    Pad a list of [T_i, D] tensors into [B, max_T, D].

    Returns:
        padded:
            [B, max_T, D]

        mask:
            [B, max_T], True for real tokens
    """

    B = len(hidden_list)
    D = hidden_list[0].shape[-1]
    device = hidden_list[0].device
    dtype = hidden_list[0].dtype

    lengths = torch.tensor(
        [x.shape[0] for x in hidden_list],
        dtype=torch.long,
        device=device,
    )

    max_T = int(lengths.max().item())

    padded = torch.full(
        (B, max_T, D),
        fill_value=padding_value,
        dtype=dtype,
        device=device,
    )

    mask = torch.zeros(
        (B, max_T),
        dtype=torch.bool,
        device=device,
    )

    for b, x in enumerate(hidden_list):
        T = x.shape[0]
        padded[b, :T] = x
        mask[b, :T] = True

    return padded, mask, lengths

In [25]:
inputs_1 = processor.apply_chat_template(
    [messages[0]],
    tokenize = True,
    add_generation_prompt = True,
    return_dict = True,
    return_tensors = "pt",
).to("cuda")

In [26]:
inputs_2 = processor.apply_chat_template(
    [messages[1]],
    tokenize = True,
    add_generation_prompt = True,
    return_dict = True,
    return_tensors = "pt",
).to("cuda")

In [27]:
inputs_3 = processor.apply_chat_template(
    [messages[2]],
    tokenize = True,
    add_generation_prompt = True,
    return_dict = True,
    return_tensors = "pt",
).to("cuda")

In [28]:
vision_hidden_1 = extract_vision_hidden(
    vlm,
    processor,
    inputs_1
)

In [29]:
vision_hidden_2 = extract_vision_hidden(
    vlm,
    processor,
    inputs_2,
)

In [30]:
vision_hidden_3 = extract_vision_hidden(
    vlm,
    processor,
    inputs_3
)

In [31]:
import gc

In [32]:

gc.collect()
torch.cuda.empty_cache()

In [33]:
vision_hidden_list = [
    vision_hidden_1,
    vision_hidden_2,
    vision_hidden_3
]

In [34]:
import numpy as np
import torch.nn as nn


def _interpolate(
    x: torch.Tensor,
    size=None,
    scale_factor=None,
    mode: str = "bilinear",
    align_corners: bool = True,
) -> torch.Tensor:
    """
    Custom interpolate to avoid INT_MAX issues.
    """

    if size is None:
        size = (
            int(x.shape[-2] * scale_factor),
            int(x.shape[-1] * scale_factor),
        )

    INT_MAX = 1610612736
    input_elements = size[0] * size[1] * x.shape[0] * x.shape[1]

    if input_elements > INT_MAX:
        chunks = torch.chunk(x, chunks=(input_elements // INT_MAX) + 1, dim=0)
        interpolated_chunks = [
            nn.functional.interpolate(
                chunk,
                size=size,
                mode=mode,
                align_corners=align_corners,
            )
            for chunk in chunks
        ]
        x = torch.cat(interpolated_chunks, dim=0)
        return x.contiguous()

    return nn.functional.interpolate(
        x,
        size=size,
        mode=mode,
        align_corners=align_corners,
    )


def interpolate_pooling_one_sample_exact_length(
    hidden,
    patch_hw,
    img_hw,
    target_len,
    pooling_func="bilinear",
    use_vggt_pe=False,
):
    """
    Pool VGGT hidden for one sample only.

    Args:
        hidden:
            [1, N_i, S, D]
            N_i is the number of real frames/images for this sample.

        patch_hw:
            (patch_h, patch_w), usually (37, 37)

        img_hw:
            (H, W), usually (518, 518)

        target_len:
            Number of Qwen visual tokens for this sample.

    Returns:
        pooled:
            [target_len, D]
    """

    patch_h, patch_w = patch_hw
    img_h, img_w = img_hw

    bs, N, S, D = hidden.shape

    if bs != 1:
        raise ValueError(f"This function expects one sample, got batch size {bs}")

    if S != patch_h * patch_w:
        raise ValueError(
            f"S={S} does not match patch_h*patch_w={patch_h * patch_w}"
        )

    if target_len <= 0:
        raise ValueError(f"target_len must be > 0, got {target_len}")

    # [1, N, S, D] -> [1, N, D, S]
    x = hidden.permute(0, 1, 3, 2)

    # [1, N, D, S] -> [N, D, patch_h, patch_w]
    x = x.reshape(N, D, patch_h, patch_w)

    if use_vggt_pe:
        x = _apply_pos_embed(x, img_w, img_h)

    # Original Spatial Forcing idea:
    # resize the spatial grid so total tokens roughly match Qwen target length.
    ratio = np.sqrt(target_len / (N * S))

    out_h = max(1, int(round(patch_h * ratio)))
    out_w = max(1, int(round(patch_w * ratio)))

    x = _interpolate(
        x,
        size=(out_h, out_w),
        mode=pooling_func,
        align_corners=True,
    )

    # [N, D, out_h, out_w]
    # -> [1, N, D, out_h*out_w]
    # -> [1, N, out_h*out_w, D]
    # -> [1, N*out_h*out_w, D]
    pooled = x.reshape(1, N, D, -1)
    pooled = pooled.permute(0, 1, 3, 2)
    pooled = pooled.reshape(1, -1, D)

    # For images, this often already exactly matches target_len.
    # For videos, Qwen may use temporal grouping, so exact match is not guaranteed.
    # Therefore, do a final 1D interpolation along token dimension if needed.
    current_len = pooled.shape[1]

    if current_len != target_len:
        pooled = F.interpolate(
            pooled.permute(0, 2, 1),  # [1, D, current_len]
            size=target_len,
            mode="linear",
            align_corners=False,
        ).permute(0, 2, 1)  # [1, target_len, D]

    return pooled.squeeze(0)

In [35]:
def masked_custom_pooling(
    hidden,
    frame_mask,
    patch_hw,
    img_hw,
    target_lengths,
    pooling_func="bilinear",
    use_vggt_pe=False,
):
    """
    Mask-aware VGGT pooling.

    Args:
        hidden:
            [B, max_N, S, D]
            VGGT hidden states after removing special tokens.

        frame_mask:
            [B, max_N]
            True for real frames/images, False for padded frames.

        target_lengths:
            [B]
            Number of Qwen visual tokens for each sample.

    Returns:
        pooled_padded:
            [B, max_T, D]

        pooled_mask:
            [B, max_T]

        pooled_list:
            List of [T_i, D]
    """

    B, max_N, S, D = hidden.shape

    pooled_list = []

    for b in range(B):
        valid_frames = frame_mask[b]

        if valid_frames.sum().item() == 0:
            raise ValueError(f"Sample {b} has no real frames/images.")

        hidden_b = hidden[b:b + 1, valid_frames, :, :]
        target_len_b = int(target_lengths[b].item())

        pooled_b = interpolate_pooling_one_sample_exact_length(
            hidden=hidden_b,
            patch_hw=patch_hw,
            img_hw=img_hw,
            target_len=target_len_b,
            pooling_func=pooling_func,
            use_vggt_pe=use_vggt_pe,
        )

        pooled_list.append(pooled_b)

    pooled_padded, pooled_mask, _ = pad_hidden_list(pooled_list)

    return pooled_padded, pooled_mask, pooled_list

In [36]:
def get_projected_vggt_for_qwen_batch(
    batch_messages,
    vision_hidden,
    vggt,
    projector,
    layers_align,
    vision_token_mask=None,
    pooling_func="bilinear",
    use_vggt_pe=False,
    mode="crop",
    max_frames=16,
    device_id="cuda",
    freeze_vggt=True,
):
    """
    Get VGGT features aligned to Qwen visual tokens.

    Args:
        batch_messages:
            Your chat-format batch.

        vision_hidden:
            Either:
                Tensor [B, T, 2560]
            or:
                List of [T_i, 2560] tensors

        vision_token_mask:
            Optional.
            If vision_hidden is padded tensor [B, T, 2560],
            this should be [B, T].
            True means real Qwen visual token.

        vggt:
            VGGT model.

        projector:
            Your projector, usually 2048 -> 2560.

    Returns:
        output dict with:
            projected_vggt:
                [B, max_T, 2560]

            vggt_token_mask:
                [B, max_T]

            projected_vggt_list:
                List of [T_i, 2560]

            frame_mask:
                [B, max_N]

            frame_lengths:
                [B]
    """

    # ------------------------------------------------------------
    # 1. Prepare Qwen visual hidden lengths
    # ------------------------------------------------------------
    if isinstance(vision_hidden, list):
        vision_hidden_padded, vision_token_mask, target_lengths = pad_hidden_list(
            vision_hidden
        )
    else:
        vision_hidden_padded = vision_hidden

        if vision_token_mask is None:
            vision_token_mask = torch.ones(
                vision_hidden_padded.shape[:2],
                dtype=torch.bool,
                device=vision_hidden_padded.device,
            )

        target_lengths = vision_token_mask.sum(dim=1)

    # ------------------------------------------------------------
    # 2. Preprocess visual batch for VGGT with frame padding
    # ------------------------------------------------------------
    vggt_imgs, frame_mask, frame_lengths = preprocess_padded_visual_batch_for_vggt(
        batch_messages=batch_messages,
        mode=mode,
        max_frames=max_frames,
        device=device_id,
    )

    # vggt_imgs:  [B, max_N, 3, 518, 518]
    # frame_mask: [B, max_N]

    # ------------------------------------------------------------
    # 3. VGGT forward
    # ------------------------------------------------------------
    if freeze_vggt:
        with torch.autocast("cuda", dtype=torch.bfloat16), torch.no_grad():
            vggt_output = vggt(vggt_imgs)
    else:
        with torch.autocast("cuda", dtype=torch.bfloat16):
            vggt_output = vggt(vggt_imgs)

    # ------------------------------------------------------------
    # 4. Extract VGGT layer and remove non-patch tokens
    # ------------------------------------------------------------
    agg_vggt_hidden = vggt_output["features"][layers_align]

    patch_start_idx = vggt_output["patch_start_idx"]
    original_img = vggt_output["images"]

    vggt_hidden = agg_vggt_hidden[:, :, patch_start_idx:, :]

    # vggt_hidden: [B, max_N, 1369, 2048]

    # ------------------------------------------------------------
    # 5. Compute VGGT patch grid
    # ------------------------------------------------------------
    H, W = original_img.shape[-2:]

    patch_h = H // vggt.patch_size
    patch_w = W // vggt.patch_size

    # usually 37, 37 for 518x518 with patch_size 14

    # ------------------------------------------------------------
    # 6. Masked pooling to Qwen visual token length
    # ------------------------------------------------------------
    pooled_vggt, vggt_token_mask, pooled_vggt_list = masked_custom_pooling(
        hidden=vggt_hidden,
        frame_mask=frame_mask,
        patch_hw=(patch_h, patch_w),
        img_hw=(H, W),
        target_lengths=target_lengths,
        pooling_func=pooling_func,
        use_vggt_pe=use_vggt_pe,
    )

    # pooled_vggt: [B, max_T, 2048]
    # vggt_token_mask: [B, max_T]

    # ------------------------------------------------------------
    # 7. Compute Spatial Forcing Loss
    # ------------------------------------------------------------
    with torch.autocast("cuda", dtype=torch.bfloat16):
        projected_qwen = projector.align_dimension(vision_hidden_padded)
        
    projected_qwen = projected_qwen * vision_token_mask.unsqueeze(-1)
    
    projected_qwen_list = []
    pooled_vggt_list = []
    
    for b in range(projected_qwen.shape[0]):
        T = int(target_lengths[b].item())
        projected_qwen_list.append(projected_qwen[b, :T].float())
        pooled_vggt_list.append(pooled_vggt[b, :T].float())
        
    loss = projector.compute_align_loss_cosine(projected_qwen_list, pooled_vggt_list)
    
    # Free up memory explicitly
    del vggt_output, vggt_hidden, vggt_imgs, pooled_vggt, vision_hidden_padded
    import gc; gc.collect()
    
    return loss


In [37]:
loss = get_projected_vggt_for_qwen_batch(
    batch_messages = messages,
    vision_hidden = vision_hidden_list,
    vggt = vggt_model,
    projector = align_projector,
    layers_align = -1,
    pooling_func = "bilinear",
    use_vggt_pe = False,
    mode = "crop",
    max_frames = 16,
    device_id = "cuda"
)

In [38]:
loss

tensor(1.0067, device='cuda:0', grad_fn=<DivBackward0>)

In [39]:
import gc
import torch

# Delete the massive Qwen intermediate outputs
if 'output_1' in globals(): del output_1
if 'output_2' in globals(): del output_2
if 'output_3' in globals(): del output_3

# Delete the old dictionary if it's still lying around
if 'out' in globals(): del out

# Force garbage collection
gc.collect()

# Force PyTorch to release the reserved VRAM back to the GPU
torch.cuda.empty_cache()

# Print current true memory usage
print(f"Allocated VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved VRAM:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


Allocated VRAM: 10.87 GB
Reserved VRAM:  11.30 GB
